# Notebook 02 — Preprocesamiento y selección de variables

**Qué hace este notebook:** define cómo se preparan los datos **sin fuga de información (data leakage)** y deja un preprocesador reutilizable (`src/preprocesamiento.py`) que usan los notebooks 03 y 04. Cada decisión se justifica con lo encontrado en el EDA.

**La regla central del enunciado:** *toda transformación que aprenda de los datos (escalado, codificación, selección de variables, umbrales) se ajusta solo con el conjunto de entrenamiento y luego se aplica a validación/prueba.*

In [1]:
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Rutas del proyecto (los notebooks viven en evaluacion 4/notebooks)
RAIZ = Path('..').resolve()
DATOS = RAIZ / 'data' / 'online_shoppers_intention.csv'
FIG = RAIZ / 'figuras'
RES = RAIZ / 'resultados'
FIG.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)

# Estilo comun a todos los graficos del proyecto
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False})
AZUL, NARANJA, GRIS, VERDE, ROJO = '#1F4E79', '#E8833A', '#8C8C8C', '#3B8E5A', '#C0392B'
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

sys.path.insert(0, str(RAIZ / 'src'))
from preprocesamiento import (cargar_datos, dividir_train_test, construir_preprocesador, matriz_conducta,
                              NUMERICAS, CATEGORICAS, BINARIAS, CONDUCTA, OBJETIVO, SEMILLA, MESES_ORDEN)

## 1. Decisiones de preprocesamiento

| # | Decisión | Por qué (evidencia del EDA) |
|---|---|---|
| D1 | **Eliminar las 125 sesiones duplicadas exactas** *antes* de dividir | Si un duplicado cae en train y otro en test, el modelo "ve" la respuesta (fuga). Además inflan el conjunto |
| D2 | **División estratificada 80/20** con semilla fija | La compra es solo ~15,6 %: sin estratificar, el test podría tener muy pocos casos positivos |
| D3 | **`log1p` en las 10 variables numéricas** | Asimetría fuerte y muchos ceros (EDA 5.2); `log1p` mantiene el 0 y reduce la cola |
| D4 | **Estandarizar** (media 0, desv. 1) las numéricas | Necesario para regresión logística y para el agrupamiento por distancias; inocuo para los árboles |
| D5 | **One-hot** para `Month`, `VisitorType`, `OperatingSystems`, `Browser`, `Region`, `TrafficType` | Son **códigos sin orden**: tratarlos como números le diría al modelo que Browser 8 > Browser 2 |
| D6 | `handle_unknown='ignore'` | Si en test aparece una categoría que no existía en train, no rompe el modelo |
| D7 | **No hay imputación** | El dataset no tiene valores faltantes (EDA 3) |
| D8 | **No eliminar atípicos** | Las colas largas son sesiones reales (visitantes muy activos); `log1p` las controla |
| D9 | **Manejar `PageValues` de forma explícita** (con y sin) | Posible fuga de información (ver EDA 3); se mide su efecto en el notebook 03 |
| D10 | **El desbalance se trata dentro del modelo** (`class_weight`) y con el **umbral de decisión**, no remuestreando | Remuestrear antes de dividir causa fuga; con `class_weight` el pipeline sigue siendo limpio |

## 2. Carga y división estratificada
Las funciones viven en `src/preprocesamiento.py` para que **todos los notebooks usen exactamente la misma división** (misma semilla).

In [2]:
df_crudo = cargar_datos(quitar_duplicados=False)
df = cargar_datos(quitar_duplicados=True)
print(f'Sesiones originales: {len(df_crudo):,} | sin duplicados: {len(df):,} (D1 elimino {len(df_crudo) - len(df)})')

X_train, X_test, y_train, y_test = dividir_train_test(df)
tabla = pd.DataFrame({
    'sesiones': [len(X_train), len(X_test)],
    'compras': [int(y_train.sum()), int(y_test.sum())],
    'tasa de compra': [y_train.mean(), y_test.mean()],
}, index=['train (80 %)', 'test (20 %)'])
tabla

Sesiones originales: 12,330 | sin duplicados: 12,205 (D1 elimino 125)


,sesiones,compras,tasa de compra
train (80 %),9764,1526,0.156
test (20 %),2441,382,0.156


La división conserva la proporción de compra (**15,6 %** en train y en test): **9.764 sesiones de entrenamiento (1.526 compras)** y **2.441 de prueba (382 compras)**. Con 382 positivos en test, los intervalos de confianza de las métricas serán anchos (±4–5 puntos en el recall): hay que leer las diferencias pequeñas entre modelos con cautela.

## 3. El preprocesador (ColumnTransformer)

In [3]:
prep = construir_preprocesador(incluir_page_values=True)
prep.fit(X_train)                                  # SOLO con entrenamiento
Xt_train = prep.transform(X_train)
Xt_test = prep.transform(X_test)                   # el test solo se TRANSFORMA

nombres = prep.get_feature_names_out()
print(f'Variables originales: {X_train.shape[1]} -> columnas tras el preprocesamiento: {Xt_train.shape[1]}')
print('Ejemplo de columnas:', list(nombres[:12]), '...')
pd.DataFrame(Xt_train[:5, :10], columns=nombres[:10])

Variables originales: 17 -> columnas tras el preprocesamiento: 73
Ejemplo de columnas: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month_Aug', 'Month_Dec'] ...


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay
0,1.611,1.263,0.910,-0.466,0.769,0.475,-0.462,-0.831,2.376,-0.319
1,-0.932,-0.984,-0.481,-0.466,0.474,0.396,-0.462,-0.766,2.307,-0.319
2,-0.932,-0.984,-0.481,-0.466,-0.241,0.009,-0.462,-0.281,-0.495,3.628
3,0.340,0.584,-0.481,-0.466,-1.003,0.153,-0.462,-0.273,-0.495,-0.319
4,-0.932,-0.984,-0.481,-0.466,-0.066,0.689,-0.462,-0.625,-0.495,-0.319


In [4]:
# Comprobacion anti-fuga: el escalador aprendio SOLO de train
num_train = Xt_train[:, :len(NUMERICAS)]
num_test = Xt_test[:, :len(NUMERICAS)]
comprobacion = pd.DataFrame({
    'media en train': num_train.mean(axis=0),
    'desv. en train': num_train.std(axis=0),
    'media en test': num_test.mean(axis=0),
    'desv. en test': num_test.std(axis=0),
}, index=NUMERICAS)
comprobacion

,media en train,desv. en train,media en test,desv. en test
Administrative,-0.000,1.000,-0.000,0.997
Administrative_Duration,0.000,1.000,-0.002,0.994
Informational,-0.000,1.000,-0.005,0.994
Informational_Duration,-0.000,1.000,0.012,1.012
ProductRelated,-0.000,1.000,0.004,0.985
ProductRelated_Duration,-0.000,1.000,-0.010,1.003
BounceRates,-0.000,1.000,0.009,1.003
ExitRates,0.000,1.000,-0.000,1.002
PageValues,-0.000,1.000,0.006,1.010
SpecialDay,0.000,1.000,-0.011,0.990


**Cómo se lee esta tabla (la prueba de que no hay fuga):**
- En **train** las variables tienen media **0,000 y desviación 1,000 exactamente**, porque el escalador se ajustó con ellas.
- En **test** las medias son **cercanas a 0 pero no iguales** (entre −0,011 y +0,012) y las desviaciones rondan 1 (0,985 a 1,012). Eso es justo lo esperado: el test se **transformó con la media y desviación de train**, sin "mirar" sus propios datos. Si saliera exactamente 0 y 1, habríamos escalado el test por separado (fuga).
- Tras el preprocesamiento las 17 variables originales se vuelven **73 columnas** (10 numéricas, 62 columnas one-hot y la binaria `Weekend`).